# Notebook 06 — Deep Learning Training (GPU — RunPod)

**Purpose:** Fine-tune pretrained segmentation models to beat the RF baseline.

This notebook has two parts:
1. **CPU preparation** — package data, generate training script, create configs
2. **GPU execution** — run on RunPod (or locally if GPU available)

**Models (in priority order):**
- Run 1: U-Net + ResNet34 (ImageNet) + Focal Loss
- Run 2: DeepLabV3+ + ResNet50 (ImageNet)
- Run 3: U-Net + ResNet34 on NLDSAR denoised tiles
- Run 4: U-Net + SAR-specific pretrained encoder (if available)
- Run 5: Larger model (ResNet101 / EfficientNet-B4)

## Part 1: CPU Preparation

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import tarfile
from pathlib import Path

from src.utils import PROCESSED_DIR, SPLITS_DIR, MODELS_DIR, PREDICTIONS_DIR, get_logger

log = get_logger('06_dl_training')

In [ ]:
# Package dataset for upload to RunPod
archive_path = PROCESSED_DIR.parent / 'titan_sar_dataset.tar.gz'

if not archive_path.exists():
    log.info("Packaging dataset for GPU upload...")
    with tarfile.open(archive_path, 'w:gz') as tar:
        tar.add(PROCESSED_DIR / 'sar_tiles', arcname='processed/sar_tiles')
        tar.add(PROCESSED_DIR / 'label_tiles', arcname='processed/label_tiles')
        if (PROCESSED_DIR / 'nldsar_tiles').exists():
            tar.add(PROCESSED_DIR / 'nldsar_tiles', arcname='processed/nldsar_tiles')
        tar.add(PROCESSED_DIR / 'class_weights.json', arcname='processed/class_weights.json')
        tar.add(SPLITS_DIR / 'split_v1.json', arcname='splits/split_v1.json')
    
    size_mb = archive_path.stat().st_size / 1e6
    log.info(f"Dataset archive: {archive_path} ({size_mb:.1f} MB)")
else:
    log.info(f"Archive already exists: {archive_path}")

### Training Script

The standalone training script is at `src/train.py`. Below we write it.

In [ ]:
%%writefile ../src/train.py
"""Standalone training script for Titan SAR segmentation — runs on GPU."""

import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import segmentation_models_pytorch as smp
import yaml
import albumentations as A

# ── Dataset ─────────────────────────────────────────────────────────────
class TitanDataset(torch.utils.data.Dataset):
    def __init__(self, tile_ids, sar_dir, label_dir, transform=None):
        self.tile_ids = sorted(tile_ids)
        self.sar_dir = Path(sar_dir)
        self.label_dir = Path(label_dir)
        self.transform = transform

    def __len__(self):
        return len(self.tile_ids)

    def __getitem__(self, idx):
        tid = self.tile_ids[idx]
        sar = np.load(self.sar_dir / f"{tid}.npy").astype(np.float32)
        label = np.load(self.label_dir / f"{tid}.npy").astype(np.int64)

        if self.transform:
            t = self.transform(image=sar, mask=label)
            sar, label = t["image"], t["mask"]

        sar = torch.from_numpy(sar).unsqueeze(0) if isinstance(sar, np.ndarray) else sar.unsqueeze(0)
        label = torch.from_numpy(label) if isinstance(label, np.ndarray) else label

        # Replicate single channel to 3 for pretrained encoders
        sar = sar.expand(3, -1, -1)

        return sar, label

# ── Loss functions ──────────────────────────────────────────────────────
def get_loss(loss_name, class_weights=None, device='cuda'):
    if loss_name == 'focal':
        return smp.losses.FocalLoss(mode='multiclass')
    elif loss_name == 'dice':
        return smp.losses.DiceLoss(mode='multiclass')
    elif loss_name == 'ce':
        w = torch.tensor(class_weights, dtype=torch.float32).to(device) if class_weights else None
        return nn.CrossEntropyLoss(weight=w, ignore_index=255)
    elif loss_name == 'focal+dice':
        focal = smp.losses.FocalLoss(mode='multiclass')
        dice = smp.losses.DiceLoss(mode='multiclass')
        return lambda pred, target: focal(pred, target) + dice(pred, target)
    else:
        raise ValueError(f"Unknown loss: {loss_name}")

# ── Metrics ─────────────────────────────────────────────────────────────
def compute_iou(pred, target, num_classes, ignore_index=255):
    ious = []
    for c in range(num_classes):
        pred_c = (pred == c)
        target_c = (target == c) & (target != ignore_index)
        inter = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        if union > 0:
            ious.append(inter / union)
    return np.mean(ious) if ious else 0.0

# ── Training loop ──────────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_targets.append(masks.cpu())
    
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    miou = compute_iou(all_preds, all_targets, num_classes)
    return total_loss / len(loader.dataset), miou

# ── Main ────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, required=True)
    parser.add_argument('--data-dir', type=str, default='data')
    parser.add_argument('--run-name', type=str, default=None)
    parser.add_argument('--device', type=str, default='cuda')
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    data_dir = Path(args.data_dir)
    run_name = args.run_name or Path(args.config).stem
    device = torch.device(args.device if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Config: {cfg}")

    # Load splits
    split_path = data_dir / 'splits' / 'split_v1.json'
    with open(split_path) as f:
        split_map = json.load(f)

    sar_dir = data_dir / 'processed' / 'sar_tiles'
    label_dir = data_dir / 'processed' / 'label_tiles'

    train_ids = [k for k, v in split_map.items() if v == 'train']
    val_ids = [k for k, v in split_map.items() if v == 'val']
    test_ids = [k for k, v in split_map.items() if v == 'test']

    # Augmentations
    train_aug = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.GaussNoise(var_limit=(0.001, 0.01), p=0.3),
    ])

    train_ds = TitanDataset(train_ids, sar_dir, label_dir, transform=train_aug)
    val_ds = TitanDataset(val_ids, sar_dir, label_dir)
    test_ds = TitanDataset(test_ids, sar_dir, label_dir)

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                              num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False,
                            num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False,
                             num_workers=4, pin_memory=True)

    # Model
    model = smp.create_model(
        cfg['architecture'],
        encoder_name=cfg['encoder'],
        encoder_weights=cfg.get('encoder_weights', 'imagenet'),
        in_channels=3,  # replicated single channel
        classes=cfg['classes'],
    ).to(device)

    # Loss
    weights_path = data_dir / 'processed' / 'class_weights.json'
    class_weights = None
    if weights_path.exists():
        with open(weights_path) as f:
            class_weights = json.load(f).get('weights_list')

    criterion = get_loss(cfg['loss'], class_weights, device)

    # Optimizer
    if cfg.get('optimizer', 'adamw') == 'adamw':
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])

    # Scheduler
    if cfg.get('scheduler') == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
    else:
        scheduler = None

    # TensorBoard
    writer = SummaryWriter(f'runs/{run_name}')

    # Training
    best_val_iou = 0
    models_dir = Path('models')
    models_dir.mkdir(exist_ok=True)
    predictions_dir = Path('data/predictions')
    predictions_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(cfg['epochs']):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_iou = evaluate(model, val_loader, criterion, device, cfg['classes'])

        if scheduler:
            scheduler.step()

        elapsed = time.time() - t0
        print(f"Epoch {epoch+1:3d}/{cfg['epochs']} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val mIoU: {val_iou:.4f} | Time: {elapsed:.1f}s")

        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('mIoU/val', val_iou, epoch)
        writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)

        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save(model.state_dict(), models_dir / f'{run_name}_best.pth')
            print(f"  -> New best model saved (mIoU={val_iou:.4f})")

    # Final evaluation on test set
    model.load_state_dict(torch.load(models_dir / f'{run_name}_best.pth', weights_only=True))
    test_loss, test_iou = evaluate(model, test_loader, criterion, device, cfg['classes'])
    print(f"\nTest Loss: {test_loss:.4f} | Test mIoU: {test_iou:.4f}")

    # Save metrics
    metrics = {
        'run_name': run_name,
        'config': cfg,
        'best_val_iou': best_val_iou,
        'test_iou': test_iou,
        'test_loss': test_loss,
    }
    with open(models_dir / f'{run_name}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    # Save test predictions
    model.eval()
    all_preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            preds = model(images).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
    all_preds = np.concatenate(all_preds)
    np.save(predictions_dir / f'{run_name}_test_predictions.npy', all_preds)

    writer.close()
    print(f"\nDone. Results saved to models/{run_name}_*")

if __name__ == '__main__':
    main()

## Part 2: GPU Execution

### RunPod Setup Instructions

```bash
# 1. Upload dataset archive
scp data/titan_sar_dataset.tar.gz runpod:/workspace/

# 2. On RunPod:
cd /workspace
tar xzf titan_sar_dataset.tar.gz -C data/
pip install -r requirements_gpu.txt

# 3. Run training
# Run 1: U-Net + ResNet34
python src/train.py --config configs/unet_resnet34.yaml --run-name unet_r34_focal

# Run 2: DeepLabV3+ + ResNet50
python src/train.py --config configs/deeplabv3_resnet50.yaml --run-name dlv3_r50_focal

# 4. Download results
scp runpod:/workspace/models/*.pth models/
scp runpod:/workspace/models/*.json models/
scp -r runpod:/workspace/runs/ runs/
scp runpod:/workspace/data/predictions/*.npy data/predictions/
```

In [ ]:
# If running locally with GPU, can train directly:
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print("\nYou can run training cells below.")
else:
    print("No GPU detected. Use RunPod or other GPU cloud.")
    print("See RunPod setup instructions above.")

## Post-Training Analysis

Run these cells after GPU training completes and results are downloaded.

In [ ]:
# Compare all runs
import glob

metric_files = sorted(MODELS_DIR.glob('*_metrics.json'))
print(f"Found {len(metric_files)} metric files\n")

comparison = []
for mf in metric_files:
    with open(mf) as f:
        m = json.load(f)
    
    run_name = m.get('run_name', mf.stem.replace('_metrics', ''))
    
    if 'test' in m and 'mean_iou' in m['test']:
        # RF baseline format
        comparison.append({
            'run': run_name,
            'test_iou': m['test']['mean_iou'],
            'test_acc': m['test']['overall_accuracy'],
        })
    elif 'test_iou' in m:
        # DL format
        comparison.append({
            'run': run_name,
            'test_iou': m['test_iou'],
            'best_val_iou': m.get('best_val_iou', None),
        })

if comparison:
    import pandas as pd
    comp_df = pd.DataFrame(comparison).sort_values('test_iou', ascending=False)
    print(comp_df.to_string(index=False))
else:
    print("No results yet. Run training first.")